# 2-stage playground — inference + evaluation (DIVA-HisDB & U-DIADS-TL, merged)

Loads the trained detector for `(BACKBONE, FAMILY, SUBSET)`, runs stage-1 boxes +
stage-2 crop segmentation on a page, visualises, then scores the full split with the
official per-family tool. Replaces the old `99_playground_cb55` + `99_playground_u_diads`.

## Parameters

In [1]:
# ==================== PARAMETERS (identical block in every 02_2stage notebook) ====================
# Switch the whole notebook by editing these three. BACKBONE is any key from
# train_detector.BACKBONES; FAMILY/SUBSET pick the dataset subset.
BACKBONE = "dinov2"      # resnet34 resnet50 convnext convnext_dinov3
                          # vit_small_patch16_224.augreg_in21k vit_small_patch16_dinov3
                          # dinov2 dinov2_reg am-radio
FAMILY   = "diva"        # "diva" | "udiads"
SUBSET   = "CB55"        # diva: CB55 CS18 CS863   |   udiads: Latin14396 Latin2 Syr341
IMGSZ    = 1280           # detector inference/training size (multiple of 32)
CONF     = 0.25           # detector confidence threshold
CROP_IMGSZ = 640          # stage-2 crop-segmenter size

# DETECTOR toggles which single-page viz cell below actually runs.
# "yolo"   -> the 2-stage BACKBONE/FAMILY/SUBSET pipeline above (unchanged).
# "rfdetr" -> the standalone RF-DETR checkpoint trained in 20_detection/rfdetr/
#             (CB55 only, single-stage full-page box detector, no crop-segmenter).
#             BACKBONE/CROP_IMGSZ are ignored for it.
DETECTOR = "rfdetr"        # "yolo" | "rfdetr"

## Resolve paths for the chosen backbone / subset

In [2]:
import sys, torch
from pathlib import Path
sys.path.insert(0, ".")
import train_detector

assert BACKBONE in train_detector.BACKBONES, f"unknown BACKBONE {BACKBONE!r}"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# (FAMILY, SUBSET) -> detection data config + trained-detector checkpoint path.
DATA_CFG = Path(f"configs/data/{('diva_'+SUBSET.lower()) if FAMILY=='diva' else ('udiads_'+SUBSET.lower())}_detect.yaml")
FAM_DIR = "diva-hisdb" if FAMILY == "diva" else "u-diads-tl"
RUN = f"yolov8_{BACKBONE}_{FAMILY}_{SUBSET}"
DET_WEIGHTS = Path(f"../../80_models/02_2stage/{FAM_DIR}/detection/{RUN}/weights/best.pt")
print("device      :", DEVICE)
print("data config :", DATA_CFG, "(exists)" if DATA_CFG.exists() else "(MISSING)")
print("detector    :", DET_WEIGHTS, "(exists)" if DET_WEIGHTS.exists() else "(train it first)")
# Absolute data YAML (fixes Ultralytics resolving `path:` vs datasets_dir).
DATA = train_detector.resolve_data(DATA_CFG) if DATA_CFG.exists() else str(DATA_CFG)

# RF-DETR checkpoint from 20_detection/rfdetr/train_rfdetr.py -- CB55 only.
RF_DETR_WEIGHTS = Path("../../20_detection/rfdetr/output/checkpoint_best_regular.pth")
if DETECTOR == "rfdetr":
    if not (FAMILY == "diva" and SUBSET == "CB55"):
        print(f"[WARN] RF-DETR was only trained on diva/CB55, not {FAMILY}/{SUBSET} -- "
              "the viz cell below will still try to load it against this split's images.")
    print("rfdetr      :", RF_DETR_WEIGHTS, "(exists)" if RF_DETR_WEIGHTS.exists() else "(train it first, see 20_detection/rfdetr/train_rfdetr.py)")


device      : cuda
data config : configs/data/diva_cb55_detect.yaml (exists)
detector    : ../../80_models/02_2stage/diva-hisdb/detection/yolov8_dinov2_diva_CB55/weights/best.pt (exists)
rfdetr      : ../../20_detection/rfdetr/output/checkpoint_best_regular.pth (exists)


## Load detector + a crop-segmenter checkpoint

In [ ]:
# Each detector is loaded only when selected -- the YOLO checkpoints and the rfdetr
# package have disjoint (and partly unsatisfiable) import requirements, so loading
# both unconditionally makes one break the other.
detector = None
rfdetr_model = None

# Crop segmenter: pick a trained stage-2 seg checkpoint (see train_crops.ipynb).
# Stage-2 only applies to the YOLO pipeline; RF-DETR is single-stage (boxes only).
SEG_WEIGHTS = Path(f"../../80_models/02_2stage/{FAM_DIR}/crop_seg/seg_{BACKBONE}_{FAMILY}_{SUBSET}/best.pth")

if DETECTOR == "yolo":
    from ultralytics import YOLO
    # Reload the trained checkpoint (backbone classes are registered on import of
    # train_detector, so hier_encoder SFP checkpoints unpickle cleanly).
    train_detector.build_model  # ensure registration side-effects have run
    detector = YOLO(str(DET_WEIGHTS)) if DET_WEIGHTS.exists() else None
    print("seg weights :", SEG_WEIGHTS, "(exists)" if SEG_WEIGHTS.exists() else "(train it first)")
elif DETECTOR == "rfdetr":
    # RF-DETR: single-stage full-page detector, no crop-segmenter (20_detection/rfdetr/).
    from rfdetr import RFDETRBase
    if RF_DETR_WEIGHTS.exists():
        rfdetr_model = RFDETRBase(pretrain_weights=str(RF_DETR_WEIGHTS), num_classes=1)
        print("loaded RF-DETR from", RF_DETR_WEIGHTS)
    else:
        print("RF-DETR checkpoint missing, train it first:", RF_DETR_WEIGHTS)
else:
    raise ValueError(f"unknown DETECTOR {DETECTOR!r}, expected 'yolo' or 'rfdetr'")

## Single page: detect → crop-segment → visualise

In [ ]:
import random, cv2, numpy as np, matplotlib.pyplot as plt
import yaml as _yaml
d = _yaml.safe_load(open(DATA))
img_dir = (Path(d["path"]) / d["test"]).resolve()
stems = sorted(p.stem for p in img_dir.glob("*.*")) if img_dir.exists() else []
print(len(stems), "test images in", img_dir)

if DETECTOR == "rfdetr":
    if rfdetr_model and stems:
        import supervision as sv
        from PIL import Image
        stem = random.choice(stems)
        img_path = next(img_dir.glob(stem + ".*"))
        image = Image.open(img_path)
        detections = rfdetr_model.predict(image, threshold=CONF)
        annotated = sv.BoxAnnotator().annotate(image.copy(), detections)
        plt.figure(figsize=(14, 10)); plt.imshow(annotated)
        plt.title(f"rfdetr · {FAMILY}/{SUBSET} · {stem} · {len(detections)} lines"); plt.axis("off"); plt.show()
    else:
        print("Train RF-DETR first (20_detection/rfdetr/train_rfdetr.py).")
elif detector and stems:
    stem = random.choice(stems)
    img_path = next(img_dir.glob(stem + ".*"))
    res = detector.predict(str(img_path), imgsz=IMGSZ, conf=CONF, verbose=False)[0]
    vis = res.plot()
    plt.figure(figsize=(14, 10)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"{BACKBONE} · {FAMILY}/{SUBSET} · {stem} · {len(res.boxes)} lines"); plt.axis("off"); plt.show()
else:
    print("Train the detector first (train_detection_colab.ipynb / run_train_all.sh).")

## Full-split evaluation (2-stage → PAGE-XML → official score)

Uses `evaluate.py`'s CLI: `predict` runs detector+segmenter over the split into PAGE-XML,
then the per-family `score-*` subcommand computes the official metrics (DIVA Java tool /
U-DIADS instance metric).

In [ ]:
# Full-split predict -> PAGE-XML -> official score.
#
# Both branches run the SAME pipeline shape, which is what makes the numbers
# comparable: detect boxes -> filter to the main TextRegion from the GT XML ->
# merge duplicate rows -> crop-segment each box into a polygon+baseline -> XML.
#
# The region filter is not optional: TASK-2 GT only annotates lines inside that
# region, while our COCO set also labels marginal gloss lines outside it. Skipping
# it turns every correct gloss detection into a false positive (measured on
# RF-DETR: LineIU 0.181 unfiltered vs 0.968 filtered).
if DETECTOR == "yolo":
    XML_OUT = Path(f"/tmp/eval_{RUN}/pred_xml"); XML_OUT.mkdir(parents=True, exist_ok=True)
    GT_XML_DIR = Path("../../00_data/DIVA-HisDB/CB55/PAGE-gt-CB55-TASK-2/TASK-2/public-test").resolve()
    cmd = f"""python evaluate.py predict \
  --dataset-family {FAMILY} \
  --yolo-weights {DET_WEIGHTS} \
  --segm-weights {SEG_WEIGHTS} \
  --test-img-dir {img_dir} \
  --test-xml-dir {GT_XML_DIR} \
  --out-xml-dir {XML_OUT}"""
    print(cmd)
    # !{cmd}    # uncomment to run (needs trained detector + segmenter)
else:
    # RF-DETR: constants (checkpoint, segmenter, thresholds) live at the top of
    # the script; it writes PAGE-XML, runs the DIVA Java evaluator, and updates
    # 20_detection/results.csv.
    cmd = "python ../../20_detection/rfdetr/eval_diva_java.py"
    print(cmd)
    # !{cmd}    # uncomment to run (~2 min: 10 pages x detect + crop-segment)